# Phase 2: Redis Fundamentals

**Roadmap position:**
```
Caching Concept (done in Phase 1)
      ↓
>>> Redis Fundamentals (you are here) <<<
      ↓
SWE Use Cases
      ↓
System Design
      ↓
AI/ML Use Cases
      ↓
Agentic AI Use Cases
      ↓
Production Deployment
```

**Goal of this notebook:** go from "I have never touched Redis" to being able to
confidently use every core data structure, understand *why* each one exists,
and finish by building **Project 1: Simple Cache Server** (FastAPI → Redis → Fake DB).

Every code cell in this notebook is connected to a **real, running Redis server** —
these are not fake/simulated outputs. You are seeing exactly what Redis returns.


## 1. What is Redis, really?

Redis = **RE**mote **DI**ctionary **S**erver.

Think of it as a giant Python dictionary that:
- lives in **RAM** (so reads/writes are microseconds, not milliseconds like a disk-based DB)
- is **shared** across all your app servers (unlike an in-memory Python dict, which only exists inside one process)
- supports more than just plain key→value — it has purpose-built data structures (Hash, List, Set, Sorted Set) that map directly to real engineering problems

**Mental model for this notebook:**

| Normal DB (Postgres/MySQL) | Redis |
|---|---|
| Stored on disk | Stored in RAM |
| Slow-ish (ms) | Extremely fast (µs) |
| Complex queries, joins | Simple key-based lookups |
| Source of truth | Temporary/fast-access layer |
| Survives restarts by default | Data is volatile unless you configure persistence |

Redis is not a *replacement* for your database. It sits **in front of** your database as a speed layer.


## 2. Installing Redis (on your own machine)

You need TWO things: the **Redis server** (the actual database) and a **Python client library** (`redis-py`) so your Python code can talk to it.

### Option A — Docker (recommended, works identically on Windows/Mac/Linux)
```bash
docker run -d --name redis-dev -p 6379:6379 redis:7
```

### Option B — Native install

**Ubuntu/Debian/WSL:**
```bash
sudo apt-get update
sudo apt-get install -y redis-server
sudo service redis-server start
```

**Mac (Homebrew):**
```bash
brew install redis
brew services start redis
```

**Windows:** native Redis isn't officially supported — use Docker (Option A) or WSL + Option B.

### Python client
```bash
pip install redis
```

### Verify it's alive
```bash
redis-cli ping
# should print: PONG
```

In *this* notebook, Redis is already installed and running on `localhost:6379` — that's what the next cell connects to.


In [4]:
!pip install redis

^C



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import redis
import time
import json
import warnings
warnings.filterwarnings("ignore")  # keep output clean for learning -- not for production code

# decode_responses=True means Redis returns Python strings instead of raw bytes
r = redis.Redis(host="localhost", port=6379, decode_responses=True)

print("Connected:", r.ping())
r.flushall()  # start with a clean slate for this demo
print("Database cleared. Ready to go.")


KeyboardInterrupt: 

## 3. Data Structure 1 — String

The simplest type. One key → one value (text, number, or JSON-serialized object).

**Core commands:** `SET`, `GET`, `DEL`, `EXPIRE`, `TTL`, `INCR`, `SETEX` (classic name for `SET` + expiry combined — you'll see it a lot in Redis docs and older tutorials)

**When you'd actually use this:**
- Caching a single computed value (an LLM answer, an API response, a rendered HTML fragment)
- Counters (page views, API call counts, retry counts)
- Feature flags / config values


In [6]:
# --- SET / GET / DEL ---
r.set("user:1:name", "Emon")
print("GET user:1:name ->", r.get("user:1:name"))

r.delete("user:1:name")
print("After DEL, GET returns:", r.get("user:1:name"))  # None = key doesn't exist

# --- TTL / EXPIRE: the heart of caching ---
r.set("otp:emon", "482913")
r.expire("otp:emon", 5)  # this key self-destructs in 5 seconds
print("TTL right after setting (seconds left):", r.ttl("otp:emon"))

# --- set() with ex=... : set + expire in one atomic command (preferred in real code) ---
r.set("otp:emon2", "731205", ex=5)
print("SETEX-style TTL:", r.ttl("otp:emon2"))

print("Waiting 6 seconds for expiration...")
time.sleep(6)
print("Value after expiry:", r.get("otp:emon"))  # gone -> None


KeyboardInterrupt: 

In [ ]:
# --- INCR: atomic counters (thread-safe, no race conditions) ---
r.delete("api_calls:user_1")

for _ in range(5):
    r.incr("api_calls:user_1")

print("Total API calls:", r.get("api_calls:user_1"))

# INCRBY for custom step, DECR for going down
r.incrby("api_calls:user_1", 10)
print("After +10:", r.get("api_calls:user_1"))

Total API calls: 5
After +10: 15


**Why `INCR` matters more than it looks:** if two requests hit `api_calls:user_1` at the exact same
millisecond, a naive "read value, add 1, write back" in your Python code can lose an update (a race condition).
Redis's `INCR` is **atomic** — Redis itself guarantees no two increments overlap. This exact pattern is
the foundation of **rate limiting** (Phase 3).


## 4. Data Structure 2 — Hash

A Hash is a key that maps to a **group of field-value pairs** — basically a mini dictionary inside Redis.
Instead of storing a JSON blob as one String (which forces you to re-write the *entire* object to change
one field), a Hash lets you update a single field directly.

**Core commands:** `HSET`, `HGET`, `HGETALL`, `HDEL`, `HINCRBY`

**When you'd actually use this:** representing an object — a user profile, a session, a product —
where you need to read/update individual fields without touching the rest.


In [ ]:
# Representing a user session as a Hash
r.delete("session:abc123")

r.hset("session:abc123", mapping={
    "user_id": "1",
    "username": "emon",
    "role": "engineer",
    "logged_in_at": "2026-07-09T10:00:00"
})

print("Full session:", r.hgetall("session:abc123"))
print("Just the username:", r.hget("session:abc123", "username"))

# Update ONE field without touching the rest -- this is the whole point of Hash
r.hset("session:abc123", "role", "senior_engineer")
print("After updating role:", r.hgetall("session:abc123"))

r.hdel("session:abc123", "logged_in_at")
print("After removing one field:", r.hgetall("session:abc123"))


Full session: {'user_id': '1', 'username': 'emon', 'role': 'engineer', 'logged_in_at': '2026-07-09T10:00:00'}
Just the username: emon
After updating role: {'user_id': '1', 'username': 'emon', 'role': 'senior_engineer', 'logged_in_at': '2026-07-09T10:00:00'}
After removing one field: {'user_id': '1', 'username': 'emon', 'role': 'senior_engineer'}


**String vs Hash — the decision rule:**
- Storing one opaque blob you always read/write whole? → **String** (e.g. `json.dumps(obj)`)
- Storing an object where you need to touch individual fields independently, often and cheaply? → **Hash**


## 5. Data Structure 3 — List

An ordered collection of values — think Python's `list`, but shared across processes.
Push to either end, pop from either end.

**Core commands:** `LPUSH`, `RPUSH`, `LPOP`, `RPOP`, `LRANGE`, `LLEN`

**When you'd actually use this:** queues. A "producer" pushes jobs in, a "consumer" pops jobs out.
This is the backbone of background task queues (e.g. Celery uses Redis as a broker exactly this way).


In [ ]:
# Simulating a simple job queue
r.delete("job_queue")

# Producer pushes 3 jobs
r.rpush("job_queue", "send_email:user_1")
r.rpush("job_queue", "generate_report:user_2")
r.rpush("job_queue", "resize_image:user_3")

print("Full queue (left=oldest, right=newest):", r.lrange("job_queue", 0, -1))
print("Queue length:", r.llen("job_queue"))

# Consumer pops jobs FIFO (First In, First Out) using LPOP
print("\n--- Consumer processing jobs ---")
while r.llen("job_queue") > 0:
    job = r.lpop("job_queue")
    print("Processing:", job)

print("\nQueue empty now:", r.lrange("job_queue", 0, -1))


Full queue (left=oldest, right=newest): ['send_email:user_1', 'generate_report:user_2', 'resize_image:user_3']
Queue length: 3

--- Consumer processing jobs ---
Processing: send_email:user_1
Processing: generate_report:user_2
Processing: resize_image:user_3

Queue empty now: []


**FIFO vs LIFO:** `RPUSH` + `LPOP` = FIFO (fair queue, first job in is first job out — what you almost
always want for task queues). `RPUSH` + `RPOP` = LIFO (stack — last job in is first out).
This single detail is a very common interview question.


## 6. Data Structure 4 — Set

An **unordered collection of unique values**. No duplicates allowed, ever — Redis enforces it for you.

**Core commands:** `SADD`, `SMEMBERS`, `SISMEMBER`, `SREM`, `SCARD`

**When you'd actually use this:** deduplication — unique visitors, unique tags, "has this user already
done X" checks, or set algebra (intersection/union) like "users who like both Python AND ML".


In [ ]:
r.delete("post:42:unique_viewers")

# Even if the same user "views" the post 3 times, the Set only stores them once
for user in ["emon", "sara", "emon", "raju", "emon"]:
    r.sadd("post:42:unique_viewers", user)

print("Unique viewers:", r.smembers("post:42:unique_viewers"))
print("Unique viewer count:", r.scard("post:42:unique_viewers"))
print("Has 'sara' viewed it?", r.sismember("post:42:unique_viewers", "sara"))
print("Has 'nobody' viewed it?", r.sismember("post:42:unique_viewers", "nobody"))


Unique viewers: {'emon', 'sara', 'raju'}
Unique viewer count: 3
Has 'sara' viewed it? 1
Has 'nobody' viewed it? 0


## 7. Data Structure 5 — Sorted Set (ZSET)

Like a Set (unique values), but every member has an attached **score**, and Redis keeps everything
sorted by that score automatically, in real time, with no extra work from you.

**Core commands:** `ZADD`, `ZRANGE`, `ZREVRANGE`, `ZSCORE`, `ZINCRBY`, `ZRANK`

**When you'd actually use this:** leaderboards, "top-N" anything, priority queues, and — spoiler for
Phase 3 — sliding-window rate limiting (score = timestamp).


In [ ]:
r.delete("leaderboard:quiz1")

r.zadd("leaderboard:quiz1", {
    "emon": 85,
    "sara": 92,
    "raju": 78,
    "nila": 92,
})

# Lowest to highest score
print("Ascending:", r.zrange("leaderboard:quiz1", 0, -1, withscores=True))

# Highest to lowest -- this is what a real leaderboard shows
print("Leaderboard (highest first):", r.zrevrange("leaderboard:quiz1", 0, -1, withscores=True))

# Someone gets 5 bonus points
r.zincrby("leaderboard:quiz1", 5, "emon")
print("Emon's new score:", r.zscore("leaderboard:quiz1", "emon"))

# What rank (0-indexed, from the top) is emon now?
print("Emon's rank from top:", r.zrevrank("leaderboard:quiz1", "emon"))

# Top 2 only
print("Top 2:", r.zrevrange("leaderboard:quiz1", 0, 1, withscores=True))


Ascending: [('raju', 78.0), ('emon', 85.0), ('nila', 92.0), ('sara', 92.0)]
Leaderboard (highest first): [('sara', 92.0), ('nila', 92.0), ('emon', 85.0), ('raju', 78.0)]
Emon's new score: 90.0
Emon's rank from top: 2
Top 2: [('sara', 92.0), ('nila', 92.0)]


## 8. Cheat Sheet — all Phase 2 commands in one place

| Data Structure | Commands | Real-world use case |
|---|---|---|
| **String** | `SET` `GET` `DEL` `SETEX` `EXPIRE` `TTL` `INCR` `INCRBY` | LLM response cache, counters, config/flags |
| **Hash** | `HSET` `HGET` `HGETALL` `HDEL` `HINCRBY` | User sessions, object profiles |
| **List** | `LPUSH` `RPUSH` `LPOP` `RPOP` `LRANGE` `LLEN` | Job queues, message queues |
| **Set** | `SADD` `SMEMBERS` `SISMEMBER` `SREM` `SCARD` | Dedup, unique visitor tracking, tags |
| **Sorted Set** | `ZADD` `ZRANGE` `ZREVRANGE` `ZSCORE` `ZINCRBY` `ZRANK` | Leaderboards, rate limiting, priority queues |

**Rule of thumb when picking a structure:** ask "what shape is my data, and what operation do I need to
be fast?" — not "which one have I heard of." That question is 90% of good Redis usage.


## 9. BUILD — Project 1: Simple Cache Server

Architecture from the roadmap:
```
FastAPI
 ↓
Redis
 ↓
Fake Database
```

This is the **Cache-Aside pattern** from Phase 1, now implemented for real:

1. Request comes in for `user_id`
2. Check Redis first — **cache hit**? return immediately (fast)
3. **Cache miss**? query the (slow) database, store the result in Redis with a TTL, then return it
4. Next request for the same `user_id` within the TTL window is now a cache hit

Below we build it step by step, then run it through a real HTTP-like client and measure the speed difference.


In [ ]:
def fake_slow_database(user_id: int) -> dict:
    """Simulates a slow DB call (e.g. a complex SQL join) taking 200ms."""
    time.sleep(0.2)
    return {
        "user_id": user_id,
        "name": f"User_{user_id}",
        "email": f"user{user_id}@example.com",
    }

CACHE_TTL_SECONDS = 30

def get_user_cached(user_id: int) -> dict:
    cache_key = f"user:{user_id}"

    cached = r.get(cache_key)
    if cached is not None:
        print(f"[CACHE HIT]  key={cache_key}")
        return json.loads(cached)

    print(f"[CACHE MISS] key={cache_key} -> querying database...")
    data = fake_slow_database(user_id)
    r.set(cache_key, json.dumps(data), ex=CACHE_TTL_SECONDS)
    return data


# --- Demo: first call is slow (miss), second call is fast (hit) ---
r.delete("user:7")

start = time.perf_counter()
result1 = get_user_cached(7)
t1 = time.perf_counter() - start
print(f"Result: {result1}  |  took {t1*1000:.1f} ms\n")

start = time.perf_counter()
result2 = get_user_cached(7)
t2 = time.perf_counter() - start
print(f"Result: {result2}  |  took {t2*1000:.1f} ms\n")

print(f"Speedup: {t1/t2:.0f}x faster on cache hit")


[CACHE MISS] key=user:7 -> querying database...


Result: {'user_id': 7, 'name': 'User_7', 'email': 'user7@example.com'}  |  took 200.8 ms



[CACHE HIT]  key=user:7
Result: {'user_id': 7, 'name': 'User_7', 'email': 'user7@example.com'}  |  took 1.7 ms

Speedup: 121x faster on cache hit


### Now the same logic, wrapped as an actual FastAPI service

This is the real, deployable shape of Project 1. In this notebook we call it in-process with
FastAPI's `TestClient` (so it runs without needing a separate terminal) — but this is the exact same
code you'd run with `uvicorn main:app` in production. A full standalone version of this project
(`main.py`, `requirements.txt`, `README.md`) has also been provided as separate files alongside this notebook.


In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI(title="Simple Cache Server")

@app.get("/user/{user_id}")
def get_user(user_id: int):
    return get_user_cached(user_id)

@app.delete("/cache/user/{user_id}")
def invalidate_user_cache(user_id: int):
    r.delete(f"user:{user_id}")
    return {"status": "cache invalidated", "user_id": user_id}


client = TestClient(app)

r.delete("user:99")  # clean slate

print("--- Request 1 (expect MISS) ---")
start = time.perf_counter()
resp1 = client.get("/user/99")
print("Status:", resp1.status_code, "| Body:", resp1.json(), f"| {(time.perf_counter()-start)*1000:.1f} ms")

print("\n--- Request 2, same user (expect HIT) ---")
start = time.perf_counter()
resp2 = client.get("/user/99")
print("Status:", resp2.status_code, "| Body:", resp2.json(), f"| {(time.perf_counter()-start)*1000:.1f} ms")

print("\n--- Invalidate cache manually ---")
print(client.delete("/cache/user/99").json())

print("\n--- Request 3, after invalidation (expect MISS again) ---")
start = time.perf_counter()
resp3 = client.get("/user/99")
print("Status:", resp3.status_code, "| Body:", resp3.json(), f"| {(time.perf_counter()-start)*1000:.1f} ms")


--- Request 1 (expect MISS) ---
[CACHE MISS] key=user:99 -> querying database...


Status: 200 | Body: {'user_id': 99, 'name': 'User_99', 'email': 'user99@example.com'} | 241.5 ms

--- Request 2, same user (expect HIT) ---
[CACHE HIT]  key=user:99
Status: 200 | Body: {'user_id': 99, 'name': 'User_99', 'email': 'user99@example.com'} | 2.9 ms

--- Invalidate cache manually ---
{'status': 'cache invalidated', 'user_id': 99}

--- Request 3, after invalidation (expect MISS again) ---
[CACHE MISS] key=user:99 -> querying database...


Status: 200 | Body: {'user_id': 99, 'name': 'User_99', 'email': 'user99@example.com'} | 202.8 ms


**What you just proved to yourself, with real measurements:**
- Cache miss ≈ 200ms (the fake DB delay)
- Cache hit ≈ well under 1ms
- `DELETE /cache/user/{id}` manually busts the cache — this is your first taste of **cache invalidation**, the topic the original roadmap calls "the most important topic" (it's covered properly in Phase 4: System Design).


## 10. Practice Exercises (do these yourself before moving to Phase 3)

1. Modify `get_user_cached` so it also tracks how many times each `user_id` has been requested,
   using `INCR` on a key like `stats:requests:user:{id}`.
2. Build a `/trending` endpoint that uses a **Sorted Set** to track the 5 most-requested user IDs
   (score = request count) — reuse the counter from exercise 1.
3. Add a `session:{token}` **Hash** to the FastAPI app representing a logged-in user, with a
   30-minute `EXPIRE`. Write a `/whoami` endpoint that reads it back.
4. Break it on purpose: set a TTL of 2 seconds on a cached user, call the endpoint, sleep 3 seconds,
   call it again — confirm in the printed logs that it correctly shows `CACHE MISS` the second time.
5. (Stretch, previews Phase 3) Use a **List** to build a naive fixed-window rate limiter: push a
   timestamp on every request to `ratelimit:{user_id}`, and reject the request if `LLEN` exceeds
   10 within the last 60 seconds.

## What's Next — Phase 3: SWE Redis

Phase 3 takes these exact same primitives (String, Hash, List, Sorted Set) and applies them to:
- **Session Management** (Hash + EXPIRE — you already did the shape of this in exercise 3)
- **Rate Limiting** (INCR + EXPIRE, or ZSET with timestamps — exercise 5 is a preview)
- **Message Queues** (List — you already built the core of this in Section 5)
- **Pub/Sub** (a new primitive — publishers broadcast, subscribers listen)

Nothing in Phase 3 is conceptually new — it's the same five tools you just used, aimed at bigger problems.
